In [1]:
# find the de genes of each cluster

print('hello')
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.pyplot import rc_context
from collections import defaultdict
import re
import os



h5ad_file = "/home/catherine/phd/projects/termites/data/znev/combined_no_norm_clustered.h5ad"
adata = sc.read_h5ad(h5ad_file)


hello


In [2]:
adata

AnnData object with n_obs × n_vars = 24252 × 14272
    obs: 'sample#', 'caste', 'n_genes_by_counts', 'total_counts', 'leiden', 'samap_annot', 'labels_only_znev_fabio', 'leiden_fabio', 'leiden_dmel_fabio', 'znev_leiden_dmel_fabio', 'paper_cell_type_annotation', 'comparison_group'
    var: 'gene_ids', 'feature_types', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'dmel_gene_ortho', 'dmel_gene_symbol'
    uns: 'annotate_cells_colors', 'annotations_1_colors', 'caste_colors', 'hvg', 'leiden', 'leiden_colors', 'leiden_dmel_fabio_colors', 'leiden_fabio_colors', 'log1p', 'neighbors', 'new_clusters_colors', 'paper_cell_type_annotation_colors', 'pca', 'rank_genes_groups', 'samap_annot_colors', 'umap', 'znev_leiden_dmel_fabio_colors'
    obsm: 'X_pca', 'X_umap', 'X_umap_fabio'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [2]:
# incorporate the gene function descriptions 
fb = pd.read_csv('/home/catherine/Downloads/best_gene_summary_fb_2025_03.tsv', sep="\t", header=8)
fb_map = dict(zip(fb["#FBgn_ID"], fb["Summary"]))

In [3]:
# Load the Excel file
file_path = "/home/catherine/phd/projects/termites/code/znev_analysis/cell_annotation/science_abk2432_table s2_markers_ref_removed.xlsx"
df = pd.read_excel(file_path)


# Initialize dictionaries with two lists: one for Curated Markers, one for Top Computed Markers (no CR, no CG)
fine_annot = defaultdict(lambda: {"curated": set(), "computed": set()})
broad_annot = defaultdict(lambda: {"curated": set(), "computed": set()})

# Iterate through the DataFrame
for _, row in df.iterrows():
    curated_markers = str(row['Curators Markers']).split(',')
    computed_markers = str(row['Top Computed Markers (no CR, no CG)']).split(',')

    curated_markers = [m.strip() for m in curated_markers if m.strip()]
    computed_markers = [m.strip() for m in computed_markers if m.strip()]

    # Add markers to the fine-level annotation dictionary
    fine_annot[row['Annotation']]['curated'].update(curated_markers)
    fine_annot[row['Annotation']]['computed'].update(computed_markers)

    # Add markers to the broad-level annotation dictionary
    broad_annot[row['Broad annotation']]['curated'].update(curated_markers)
    broad_annot[row['Broad annotation']]['computed'].update(computed_markers)

# Convert sets to lists for final output
fine_annot = {k: {'curated': list(v['curated']), 'computed': list(v['computed'])} for k, v in fine_annot.items()}
broad_annot = {k: {'curated': list(v['curated']), 'computed': list(v['computed'])} for k, v in broad_annot.items()}

In [4]:
# Invert curated dicts: symbol -> [fly cell types]
sym2fine = defaultdict(list)
for fly_ct, d in fine_annot.items():
    for sym in d.get("curated", []):
        sym2fine[sym].append(fly_ct)
sym2fine = {k: sorted(set(v)) for k, v in sym2fine.items()}

sym2broad = defaultdict(list)
for fly_ct, d in broad_annot.items():
    for sym in d.get("curated", []):
        sym2broad[sym].append(fly_ct)
sym2broad = {k: sorted(set(v)) for k, v in sym2broad.items()}

In [9]:
# computes cell markers, all genes, all castes, for both leiden clusters and annotated cell types. 

cell_type_col = "leiden"
excel_filename = "celltype_leiden_wilcoxon_markers_all_genes.xlsx"

sc.tl.rank_genes_groups(
        adata,
        groupby=cell_type_col,
        method="wilcoxon",
        use_raw=False,
    )

#groups = list(adata.obs['paper_cell_type_annotation'].cat.categories)
groups = list(adata.obs[cell_type_col].cat.categories)

#with pd.ExcelWriter('celltype_wilcoxon_markers_all_genes.xlsx') as w:
with pd.ExcelWriter(excel_filename) as w:

    for g in groups:
        df = sc.get.rank_genes_groups_df(adata, group=g)[
            ["names", "scores", "pvals", "logfoldchanges"]
        ].sort_values("scores", ascending=False)

        df = df.rename(columns={
            "names": "gene",
            "scores": "score",
            "pvals": "pvalue",
            "logfoldchanges": "logfc",
        })

        df["d_m_ortho"] = adata.var.loc[df["gene"], 'dmel_gene_ortho'].values

        df["d_m_ortho_sym"] = adata.var.loc[df["gene"], 'dmel_gene_symbol'].values

        df["function"] = df["d_m_ortho"].map(fb_map)

        symbols = adata.var.loc[df["gene"], 'dmel_gene_symbol'].values
        df["fine_annot_curated"]  = ["; ".join(sym2fine.get(sym, []))  for sym in symbols]
        df["broad_annot_curated"] = ["; ".join(sym2broad.get(sym, [])) for sym in symbols]


        df[["gene", "score", "pvalue", "logfc", "d_m_ortho", "d_m_ortho_sym", "fine_annot_curated", "broad_annot_curated", "function"]].to_excel(
            w, sheet_name=str(g), index=False
        )

/home/catherine/anaconda3/envs/sc_termites/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:458: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/home/catherine/anaconda3/envs/sc_termites/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:460: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/home/catherine/anaconda3/envs/sc_termites/lib/python3.12/site-packages/scanpy/tools/_rank_gene

In [7]:
target_castes=("soldier", "king", "queen")
reference_caste="worker"
cell_type_col="paper_cell_type_annotation"
up_reg = True
min_cells_per_group=2
outfile_prefix="ct_wilcoxon_up_reg_genes"
ortho_col="dmel_gene_ortho"
symbol_col="dmel_gene_symbol"

In [14]:
import os
import pandas as pd
import scanpy as sc

target_castes = ("soldier", "king", "queen")
reference_caste = "worker"
cell_type_col = "paper_cell_type_annotation"
min_cells_per_group = 2

outfile_prefix = "ct_wilcoxon_genes"
ortho_col = "dmel_gene_ortho"
symbol_col = "dmel_gene_symbol"

# output folders
base_dir = "ct_wilcoxon_both_directions"
up_dir = os.path.join(base_dir, "up")
down_dir = os.path.join(base_dir, "down")
os.makedirs(up_dir, exist_ok=True)
os.makedirs(down_dir, exist_ok=True)

# shared category names (same logic for up and down)
categories = [
    "soldier_total", "king_total", "queen_total",
    "unique_soldier", "unique_king", "unique_queen",
    "soldier_king_only", "soldier_queen_only", "king_queen_only",
    "all_three", "any"
]

# ---------- compute DE per cell type (once), store BOTH up + down ----------
cell_types = adata.obs[cell_type_col].unique()

de_tables = {ct: {caste: {} for caste in target_castes} for ct in cell_types}
de_sets   = {ct: {caste: {"up": set(), "down": set()} for caste in target_castes} for ct in cell_types}

for ct in cell_types:
    ad_ct = adata[adata.obs[cell_type_col] == ct].copy()

    # if too few workers in this CT, leave empties
    if (ad_ct.obs["caste"] == reference_caste).sum() < min_cells_per_group:
        continue

    for caste in target_castes:
        subset = ad_ct[ad_ct.obs["caste"].isin([caste, reference_caste])].copy()
        n_caste  = (subset.obs["caste"] == caste).sum()
        n_worker = (subset.obs["caste"] == reference_caste).sum()

        if n_caste < min_cells_per_group or n_worker < min_cells_per_group:
            continue

        subset.obs["caste_group"] = (
            (subset.obs["caste"] == caste)
            .map({True: "caste", False: "worker"})
            .astype("category")
        )

        sc.tl.rank_genes_groups(
            subset,
            groupby="caste_group",
            groups=["caste"],
            reference="worker",
            method="wilcoxon",
            pts=True
        )

        de = sc.get.rank_genes_groups_df(subset, group="caste").copy()
        de = de.rename(columns={
            "names": "gene",
            "scores": "score",
            "logfoldchanges": "logfc",
            "pvals": "pvalue",
            "pvals_adj": "pvalue_adj",
        })
        de["caste"] = caste

        # store full table (unfiltered) for this comparison
        de_tables[ct][caste]["all"] = de

        # store up/down filtered tables
        de_tables[ct][caste]["up"] = de[de["logfc"] > 0].copy()
        de_tables[ct][caste]["down"] = de[de["logfc"] < 0].copy()

        # store up/down gene sets
        de_sets[ct][caste]["up"] = set(de_tables[ct][caste]["up"]["gene"].astype(str))
        de_sets[ct][caste]["down"] = set(de_tables[ct][caste]["down"]["gene"].astype(str))


def build_ct_category_sets(direction: str):
    """direction in {'up','down'}"""
    ct_category_sets = {ct: {} for ct in cell_types}
    for ct in cell_types:
        S = de_sets[ct].get("soldier", {}).get(direction, set())
        K = de_sets[ct].get("king", {}).get(direction, set())
        Q = de_sets[ct].get("queen", {}).get(direction, set())

        all_three = S & K & Q
        SK_only = (S & K) - all_three
        SQ_only = (S & Q) - all_three
        KQ_only = (K & Q) - all_three
        S_only = S - (K | Q)
        K_only = K - (S | Q)
        Q_only = Q - (S | K)
        union_any = S | K | Q

        ct_category_sets[ct] = {
            "soldier_total": S,
            "king_total":    K,
            "queen_total":   Q,
            "unique_soldier": S_only,
            "unique_king":    K_only,
            "unique_queen":   Q_only,
            "soldier_king_only":  SK_only,
            "soldier_queen_only": SQ_only,
            "king_queen_only":    KQ_only,
            "all_three":      all_three,
            "any":            union_any,
        }
    return ct_category_sets


def export_direction(direction: str, out_dir: str):
    """
    Export 11 category workbooks for the given direction ('up' or 'down')
    into out_dir, with sheets per cell type.
    """
    ct_category_sets = build_ct_category_sets(direction)

    for cat in categories:
        out_path = os.path.join(out_dir, f"{outfile_prefix}_{direction}_{cat}.xlsx")

        with pd.ExcelWriter(out_path) as w:
            for ct in cell_types:
                genes = ct_category_sets[ct][cat]
                if not genes:
                    pd.DataFrame().to_excel(w, sheet_name=str(ct), index=False)
                    continue

                dfs = []
                for caste in target_castes:
                    df = de_tables[ct][caste].get(direction, pd.DataFrame())
                    if df is None or df.empty:
                        continue
                    dfs.append(df[df["gene"].isin(genes)])

                if not dfs:
                    pd.DataFrame().to_excel(w, sheet_name=str(ct), index=False)
                    continue

                merged = pd.concat(dfs, ignore_index=True)

                merged["d_m_ortho"] = adata.var.loc[merged["gene"], ortho_col].values
                symbols = adata.var.loc[merged["gene"], symbol_col].values
                merged["function"] = merged["d_m_ortho"].map(fb_map)
                merged["fine_annot_curated"]  = ["; ".join(sym2fine.get(sym, []))  for sym in symbols]
                merged["broad_annot_curated"] = ["; ".join(sym2broad.get(sym, [])) for sym in symbols]

                merged[
                    ["gene", "score", "pvalue", "pvalue_adj", "logfc", "caste",
                     "d_m_ortho", "function",
                     "fine_annot_curated", "broad_annot_curated"]
                ].to_excel(w, sheet_name=str(ct), index=False)

        print(f"✔ Wrote {out_path}")


# ---------- export both directions ----------
export_direction("up", up_dir)
export_direction("down", down_dir)

print(f"✔ Done: exported {len(categories)} up + {len(categories)} down workbooks into '{base_dir}/'.")


✔ Wrote ct_wilcoxon_both_directions/up/ct_wilcoxon_genes_up_soldier_total.xlsx
✔ Wrote ct_wilcoxon_both_directions/up/ct_wilcoxon_genes_up_king_total.xlsx
✔ Wrote ct_wilcoxon_both_directions/up/ct_wilcoxon_genes_up_queen_total.xlsx
✔ Wrote ct_wilcoxon_both_directions/up/ct_wilcoxon_genes_up_unique_soldier.xlsx
✔ Wrote ct_wilcoxon_both_directions/up/ct_wilcoxon_genes_up_unique_king.xlsx
✔ Wrote ct_wilcoxon_both_directions/up/ct_wilcoxon_genes_up_unique_queen.xlsx
✔ Wrote ct_wilcoxon_both_directions/up/ct_wilcoxon_genes_up_soldier_king_only.xlsx
✔ Wrote ct_wilcoxon_both_directions/up/ct_wilcoxon_genes_up_soldier_queen_only.xlsx
✔ Wrote ct_wilcoxon_both_directions/up/ct_wilcoxon_genes_up_king_queen_only.xlsx
✔ Wrote ct_wilcoxon_both_directions/up/ct_wilcoxon_genes_up_all_three.xlsx
✔ Wrote ct_wilcoxon_both_directions/up/ct_wilcoxon_genes_up_any.xlsx
✔ Wrote ct_wilcoxon_both_directions/down/ct_wilcoxon_genes_down_soldier_total.xlsx
✔ Wrote ct_wilcoxon_both_directions/down/ct_wilcoxon_genes

In [5]:
import numpy as np
import pandas as pd
import scanpy as sc

# ---------------------- CONFIG (from your cell) ----------------------
target_castes      = ("soldier", "king", "queen")
reference_caste    = "worker"
cell_type_col      = "paper_cell_type_annotation"

min_cells_per_group = 2
outfile_prefix      = "ct_wilcoxon_sig_genes"   # updated name; no Excel files
ortho_col           = "dmel_gene_ortho"
symbol_col          = "dmel_gene_symbol"

ALPHA = 0.05  # significance threshold

# Categories we will count for each cell type
CATEGORIES_BASE = [
    "soldier_total", "king_total", "queen_total",
    "unique_soldier", "unique_king", "unique_queen",
    "soldier_king_only", "soldier_queen_only", "king_queen_only",
    "all_three"
]
# We'll append an "any_*" column per direction ("any_up", "any_down")

# ---------------------- ASSERTIONS ----------------------
assert "caste" in adata.obs.columns, "adata.obs must contain a 'caste' column"
assert cell_type_col in adata.obs.columns, f"adata.obs must contain '{cell_type_col}'"
for c in target_castes + (reference_caste,):
    assert (adata.obs["caste"] == c).any(), f"No cells found for caste: {c}"

# ---------------------- DE HELPERS ----------------------
def run_pairwise_de(subset: sc.AnnData) -> pd.DataFrame:
    """
    Runs Wilcoxon DE in subset (already filtered to caste and worker),
    comparing group 'caste' vs 'worker'.
    Returns a DataFrame with columns:
      gene, score, logfc, pvalue, pvalue_adj, caste
    """
    sc.tl.rank_genes_groups(
        subset,
        groupby="caste_group",
        groups=["caste"],
        reference="worker",
        method="wilcoxon",
        pts=True
    )
    de = sc.get.rank_genes_groups_df(subset, group="caste").copy()
    de = de.rename(columns={
        "names": "gene",
        "scores": "score",
        "logfoldchanges": "logfc",
        "pvals": "pvalue",
        "pvals_adj": "pvalue_adj",
    })
    return de

def build_category_sets(S: set, K: set, Q: set) -> dict:
    """Return dict of category -> set of genes based on S/K/Q overlaps."""
    all_three = S & K & Q
    SK_only   = (S & K) - all_three
    SQ_only   = (S & Q) - all_three
    KQ_only   = (K & Q) - all_three
    S_only    = S - (K | Q)
    K_only    = K - (S | Q)
    Q_only    = Q - (S | K)
    union_any = S | K | Q

    return {
        "soldier_total": S,
        "king_total":    K,
        "queen_total":   Q,
        "unique_soldier": S_only,
        "unique_king":    K_only,
        "unique_queen":   Q_only,
        "soldier_king_only":  SK_only,
        "soldier_queen_only": SQ_only,
        "king_queen_only":    KQ_only,
        "all_three":      all_three,
        "union_any":      union_any,  # used to fill any_up/any_down later
    }

# ---------------------- RUN DE ACROSS CELL TYPES ----------------------
cell_types = adata.obs[cell_type_col].unique()

# Store DE tables (per cell type, per caste)
de_tables = {ct: {} for ct in cell_types}

for ct in cell_types:
    ad_ct = adata[adata.obs[cell_type_col] == ct].copy()

    # Need enough reference cells in this cell type
    if (ad_ct.obs["caste"] == reference_caste).sum() < min_cells_per_group:
        for caste in target_castes:
            de_tables[ct][caste] = pd.DataFrame(columns=["gene","score","logfc","pvalue","pvalue_adj","caste"])
        continue

    for caste in target_castes:
        subset = ad_ct[ad_ct.obs["caste"].isin([caste, reference_caste])].copy()
        n_caste  = int((subset.obs["caste"] == caste).sum())
        n_worker = int((subset.obs["caste"] == reference_caste).sum())
        if n_caste < min_cells_per_group or n_worker < min_cells_per_group:
            de_tables[ct][caste] = pd.DataFrame(columns=["gene","score","logfc","pvalue","pvalue_adj","caste"])
            continue

        subset.obs["caste_group"] = (subset.obs["caste"] == caste).map({True: "caste", False: "worker"}).astype("category")
        de = run_pairwise_de(subset)
        de["caste"] = caste
        de_tables[ct][caste] = de

# ---------------------- SPLIT INTO UP / DOWN SIGNIFICANT SETS ----------------------
# For each cell type, create sets of significant UP and DOWN genes per caste
de_sets_up   = {ct: {} for ct in cell_types}
de_sets_down = {ct: {} for ct in cell_types}

for ct in cell_types:
    for caste in target_castes:
        df = de_tables[ct][caste]
        if df.empty:
            de_sets_up[ct][caste]   = set()
            de_sets_down[ct][caste] = set()
            continue

        sig = df.loc[df["pvalue_adj"] < ALPHA].copy()

        up_genes   = set(sig.loc[sig["logfc"] > 0, "gene"].astype(str))
        down_genes = set(sig.loc[sig["logfc"] < 0, "gene"].astype(str))

        de_sets_up[ct][caste]   = up_genes
        de_sets_down[ct][caste] = down_genes

# ---------------------- BUILD CATEGORY SETS (UP & DOWN) ----------------------
ct_category_sets_up   = {}
ct_category_sets_down = {}

for ct in cell_types:
    # UP
    S_up = de_sets_up[ct].get("soldier", set())
    K_up = de_sets_up[ct].get("king", set())
    Q_up = de_sets_up[ct].get("queen", set())
    cats_up = build_category_sets(S_up, K_up, Q_up)
    cats_up["any_up"] = cats_up.pop("union_any")
    ct_category_sets_up[ct] = cats_up

    # DOWN
    S_dn = de_sets_down[ct].get("soldier", set())
    K_dn = de_sets_down[ct].get("king", set())
    Q_dn = de_sets_down[ct].get("queen", set())
    cats_dn = build_category_sets(S_dn, K_dn, Q_dn)
    cats_dn["any_down"] = cats_dn.pop("union_any")
    ct_category_sets_down[ct] = cats_dn

# ---------------------- MAKE COUNT TABLES (UP & DOWN) ----------------------
def make_count_table(ct_category_sets: dict, direction_any_col: str) -> pd.DataFrame:
    cols = CATEGORIES_BASE + [direction_any_col]
    index = pd.Index(cell_types, name="cell_type")
    out = pd.DataFrame(0, index=index, columns=cols, dtype=int)
    for ct in cell_types:
        if ct not in ct_category_sets:
            continue
        for c in cols:
            if c in ct_category_sets[ct]:
                out.loc[ct, c] = len(ct_category_sets[ct][c])
    return out

up_table_counts   = make_count_table(ct_category_sets_up,   "any_up")
down_table_counts = make_count_table(ct_category_sets_down, "any_down")

# ---------------------- SAVE TABLES ----------------------
up_path   = f"{outfile_prefix}__UP_counts_by_ct_category.csv"
down_path = f"{outfile_prefix}__DOWN_counts_by_ct_category.csv"

up_table_counts.to_csv(up_path)
down_table_counts.to_csv(down_path)

print(f"✔ Saved UP table:   {up_path}")
print(f"✔ Saved DOWN table: {down_path}")


✔ Saved UP table:   ct_wilcoxon_sig_genes__UP_counts_by_ct_category.csv
✔ Saved DOWN table: ct_wilcoxon_sig_genes__DOWN_counts_by_ct_category.csv
